In [17]:
print('hello world')

hello world


In [18]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [19]:
import pandas as pd
import numpy as np
import os
os.environ['USE_PYGEOS'] = '0'
from itertools import product
import math
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from joblib import dump

# from models import *
from sklearn.model_selection import train_test_split
from Models.mlp import *
import time
import geopandas as gpd

Load Dataset

In [ ]:
dataframe = pd.read_csv("../Data/unitask_data/dataset.csv")

Useful functions

In [21]:
def create_results_csv(csv_columns, csv_name):
    experiments = csv_name

    if os.path.exists(experiments):
        df_experiments = pd.read_csv(experiments)
    else:
        df_experiments = pd.DataFrame(columns=csv_columns)
        df_experiments.to_csv(experiments, index=False)

    return df_experiments

In [22]:
def check_existance_params_in_experiments(df_experiments, config_to_check):

    # Extract relevant columns from DataFrame
    relevant_columns = list(config_to_check.keys())
    relevant_df = df_experiments[relevant_columns]
    # Ensure data types match
    for col in relevant_columns:
        if col in relevant_df and col in config_to_check:
            # Convert lists to strings for comparison
            if config_to_check[col] is not None:
                if len(relevant_df[col]) > 0 and isinstance(relevant_df[col].iloc[0], list):
                    relevant_df[col] = relevant_df[col].apply(str)
                    config_to_check[col] = str(config_to_check[col])
                # Otherwise, cast to DataFrame's column type
                elif relevant_df[col].dtype == 'object':  # Handle object (or string-like) columns
                    config_to_check[col] = str(config_to_check[col])
                else:
                    try:
                        config_to_check[col] = relevant_df[col].dtype.type(config_to_check[col])
                    except ValueError:
                        # keep the original (string) value
                        pass
            else:
                config_to_check[col] = ""
                relevant_df[col].fillna("", inplace=True)

    # Convert the configuration to a DataFrame row
    config_df = pd.DataFrame([config_to_check])

    # Check if the configuration exists
    exists = any((relevant_df == config_df.iloc[0]).all(axis=1))
    return exists

In [23]:
# Determining features and target columns
columns = dataframe.columns.to_list()
soil_properties = ['pH', 'CE', 'CEC', 'OM', 'A%', 'L%', 'S%']
cols_to_drop = soil_properties + ['X', 'Y']
covariates = [col for col in columns if col not in cols_to_drop]

In [24]:
gdf = gpd.GeoDataFrame(
    dataframe,
    geometry=gpd.points_from_xy(dataframe['X'], dataframe['Y']),
    crs='EPSG:4326'
)

gdf = gdf.to_crs('EPSG:32631')

# Keep expected 47-column layout: covariates + soil properties + X + Y
ordered_cols = covariates + soil_properties + ['X', 'Y']

dataframe_proj = dataframe.copy()
dataframe_proj['X'] = gdf.geometry.x.values
dataframe_proj['Y'] = gdf.geometry.y.values

points = dataframe_proj[ordered_cols].to_numpy()
print("Points shape:", points.shape)

Points shape: (2156, 47)


In [25]:
# To train model with depths

dfs = {
        "all_depths":dataframe
        }



normalizations = [
    'StandardScaler',
    # "Without",
    # 'MinMaxScaler',

]


soil_properties_options = [['pH'], ['CE'], ['CEC'], ['OM'] ,['A%'], ['L%'], ['S%']]
                        

## Getting Cross-validation folds

### Spatial layout diagnostics
Used to pick sensible spatial parameters (block size / AHC threshold) for
the spatial CV methods below.

In [26]:
from CVs.cv_utils import build_folds_by_method

block_thresholds_km = [30, 20, 50, 100, 5, 10, 150, 200]
sp1_thresholds_km = [30, 20, 50, 100, 5, 10, 150, 200]
sp2_thresholds_km = [30, 20, 50, 100, 5, 10, 150, 200]


block_thresholds_m = [v * 1000 for v in block_thresholds_km]
sp1_thresholds_m = [v * 1000 for v in sp1_thresholds_km]
sp2_thresholds_m = [v * 1000 for v in sp2_thresholds_km]

N_FOLDS = 5
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


n_cov = len(covariates)        
n_tar = len(soil_properties) 
print(n_cov, n_tar) 

env = points[:, :n_cov]                
tar = points[:, n_cov:n_cov + n_tar]  
REGION = 'all'
folds_by_method = build_folds_by_method(
    region = REGION,
    points = points,
    env = env,
    tar = tar,
    n_folds=N_FOLDS,
    block_sp_threshold=block_thresholds_m,
    sp1_threshold=sp1_thresholds_m,
    sp_threshold=sp2_thresholds_m,
    random_repeats=1
)


print(sorted(folds_by_method.keys()))

for name, folds in folds_by_method.items():
    n_pts = len(folds)
    n_f = len(set(int(row[1]) for row in folds))
    print(f"{name:>6}: {n_pts} points assigned across {n_f} folds")


38 7
d:\DSM_MODULE\CVs\cv_folds\all\random
d:\DSM_MODULE\CVs\cv_folds\all\sp1
d:\DSM_MODULE\CVs\cv_folds\all\sp2
d:\DSM_MODULE\CVs\cv_folds\all\sp
Building folds by method in d:\DSM_MODULE\CVs\cv_folds\all...
  - Random CV: 1 repeats, k=5
Generating random-CV folds.
Checking for existing CSV: d:\DSM_MODULE\CVs\cv_folds\all\random\randomCV_rep0_k5.csv
  - Block CV: k=5, thresholds=[30000, 20000, 50000, 100000, 5000, 10000, 150000, 200000]
    - Threshold: 30000
Generating block-CV folds.
    - Threshold: 20000
Generating block-CV folds.
    - Threshold: 50000
Generating block-CV folds.
    - Threshold: 100000
Generating block-CV folds.
    - Threshold: 5000
Generating block-CV folds.
    - Threshold: 10000
Generating block-CV folds.
    - Threshold: 150000
Generating block-CV folds.
    - Threshold: 200000
Generating block-CV folds.
  - SP1 CV: k=5, thresholds=[30000, 20000, 50000, 100000, 5000, 10000, 150000, 200000]
    - Threshold: 30000
    - Threshold: 20000
    - Threshold: 50000


In [27]:
def build_group_array(dataframe, folds, id_col='ID'):
    """Map each row of `dataframe` to its assigned fold.

    The CV functions use the dataframe row index as the point ID because
    the dataset has no explicit ID column.

    `folds` contains [point_ID, fold_index, ...].
    """

    fold_map = {
        int(row[0]): int(row[1])
        for row in folds
    }

    # No explicit ID column in the dataset.
    # The row index is the point ID.
    ids = np.arange(len(dataframe))

    try:
        groups = np.array([
            fold_map[int(i)]
            for i in ids
        ])
    except KeyError as e:
        raise ValueError(
            f"Point ID {e} from dataframe has no fold assignment."
        )

    return groups

###  MLP

In [28]:
def custom_cross_validation_2D_DSM_MLP(region, dataframe, param_grid, normalization, model,
                                        folds, cv_method, soil_properties=soil_properties,
                                        depth='', plot_cv=True):
    """
    Grid-search + evaluate an MLP (multi-output over `soil_properties`) using a
    leave-one-fold-out scheme defined by `folds` (a [point_ID, fold_index, ...]
    list produced by randomCV.py / spatialCV.py, generated once earlier and
    shared by every model/normalization run) -- mirrors
    `custom_cross_validation_2D_DSM_ML` in trees_experiments.ipynb.

    `cv_method` is a short label ('random', 'block', 'sp1', ...) used to keep
    results and saved models separate across CV strategies.
    """
    # Ensure output directories exist
# Ensure output directories exist
    os.makedirs('../Results/evaluation_on_bestconfigs/', exist_ok=True)
    os.makedirs(f'../Results/evaluation_on_bestconfigs/{region}/{model}/', exist_ok=True)
    os.makedirs(f'../Models/{region}/{model}/{cv_method}', exist_ok=True)
    os.makedirs(f'../Results/best_params/{region}/{model}/{cv_method}', exist_ok=True)

    # Create results CSV (one per model x CV method)
    experiments = f'../Results/evaluation_on_bestconfigs/{region}/{model}/experiments_{model}_{cv_method}_2d_dm_new.csv'
    csv_columns = (['soil_properties', 'cv_method'] + ['normalization', 'depth', 'loss', 'optimizer']
                   + list(param_grid.keys()) + ['history', 'metrics', 'rmse', 'mse', 'r2', 'mae', 'time'])
    df_experiments = create_results_csv(csv_columns, experiments)

    # Selecting features / targets
    features = dataframe[covariates]
    y_all = dataframe[soil_properties]

    # Map every row to its pre-computed fold (from randomCV / spatialCV)
    groups = build_group_array(dataframe, folds)
    unique_folds = np.sort(np.unique(groups))

    # Selecting scaler classes (re-instantiated fresh for every fold, like the
    # per-fold scaler.fit_transform / scaler.transform pattern in trees_experiments)
    if normalization == 'MinMaxScaler':
        scaler_X_cls, scaler_Y_cls = MinMaxScaler, MinMaxScaler
    elif normalization == 'StandardScaler':
        scaler_X_cls, scaler_Y_cls = StandardScaler, StandardScaler
    else:
        scaler_X_cls, scaler_Y_cls = None, None

    best_rmse = float('inf')
    best_r2 = -float('inf')
    best_model = None
    best_params = None

    for params in product(*param_grid.values()):
        param_dict = dict(zip(param_grid.keys(), params))
        print(param_dict)

        config_to_check = param_dict.copy()
        config_to_check['soil_properties'] = ",".join(soil_properties)
        config_to_check['cv_method'] = cv_method
        config_to_check['normalization'] = normalization
        config_to_check['depth'] = depth

        exists = check_existance_params_in_experiments(df_experiments, config_to_check)

        if exists:
            print('yesssss exists')
            continue

        y_predict = []
        y_true = []
        histories = []
        last_model = None
        start_time = time.time()

        for fold_val in unique_folds:
            test_mask = groups == fold_val
            train_mask = ~test_mask
            train_index = np.where(train_mask)[0]
            test_index = np.where(test_mask)[0]

            X_train, X_test = features.iloc[train_index], features.iloc[test_index]
            y_train, y_test = y_all.iloc[train_index], y_all.iloc[test_index]

            scaler_X = scaler_X_cls() if scaler_X_cls else None
            scaler_Y = scaler_Y_cls() if scaler_Y_cls else None

            if scaler_X:
                X_train = pd.DataFrame(scaler_X.fit_transform(X_train), columns=covariates)
                X_test = pd.DataFrame(scaler_X.transform(X_test), columns=covariates)

            if scaler_Y:
                y_train_fit = pd.DataFrame(scaler_Y.fit_transform(y_train), columns=soil_properties)
            else:
                y_train_fit = y_train

            # Fit model on this fold's training split
            training_model, history = MLP_training(
                param_dict, X_train, y_train_fit, n_outputs=len(soil_properties), validation_data=None)
            
            last_model = training_model
            histories.append(getattr(history, 'history', history))

            model_predict = training_model.predict(X_test)
            if scaler_Y:
                model_predict = scaler_Y.inverse_transform(model_predict)

            fold_pred = np.array(model_predict)
            fold_true = np.array(y_test)

            if np.isinf(fold_pred).any():
                print(f'skipped fold {fold_val} because of inf')
                continue
            if np.isnan(fold_pred).any():
                print(f'skipped fold {fold_val} because of nan')
                continue

            y_predict.extend(fold_pred.tolist())
            y_true.extend(fold_true.tolist())

        training_time = time.time() - start_time

        if len(y_true) == 0:
            print('skipped configuration - no valid folds')
            continue

        y_true = np.array(y_true)
        y_predict = np.array(y_predict)

        # Metrics computed once, over predictions pooled across every fold
        # (same aggregation style as trees_experiments.ipynb)
        overall_mse = mean_squared_error(y_true, y_predict)
        overall_rmse = math.sqrt(overall_mse)
        overall_r2 = r2_score(y_true, y_predict)
        overall_mae = mean_absolute_error(y_true, y_predict)

        metrics = {}
        for i, target in enumerate(soil_properties):
            mae = mean_absolute_error(y_true[:, i], y_predict[:, i])
            mse = mean_squared_error(y_true[:, i], y_predict[:, i])
            rmse = np.sqrt(mse)
            r2 = r2_score(y_true[:, i], y_predict[:, i])
            metrics[target] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

        if overall_rmse < best_rmse and overall_r2 > best_r2:
            best_r2 = overall_r2
            best_rmse = overall_rmse
            best_model = last_model
            best_params = param_dict

        row = param_dict.copy()
        row.update({
            'soil_properties': ",".join(soil_properties),
            'cv_method': cv_method,
            'normalization': normalization,
            'depth': depth,
            'loss': 'mse',
            'optimizer': 'Adam',
            'rmse': overall_rmse,
            'mse': overall_mse,
            'r2': overall_r2,
            'mae': overall_mae,
            'time': training_time,
            'history': histories,
            'metrics': metrics,
        })

        df_experiments.loc[len(df_experiments)] = row
        df_experiments.to_csv(experiments, index=False)

    if best_model:
        dump(best_model, f'../Models/{region}/{model}/{cv_method}/best_{model}_{cv_method}_{depth}_[{"-".join(soil_properties)}]_{normalization}.joblib')
        best_params_df = pd.DataFrame([best_params])
        best_params_df['r2'] = best_r2
        best_params_df['rmse'] = best_rmse
        best_params_df.to_csv(
            f'../Results/best_params/{region}/{model}/best_params_2D_{model}_{cv_method}_{depth}_[{"-".join(soil_properties)}]_{normalization}.csv',
            index=False
        )


#### Training MLP

In [29]:
folds_by_method.keys()

dict_keys(['random', 'block_k5_30000', 'block_k5_20000', 'block_k5_50000', 'block_k5_100000', 'block_k5_5000', 'block_k5_10000', 'block_k5_150000', 'block_k5_200000', 'sp1_k5_30000', 'sp1_k5_20000', 'sp1_k5_50000', 'sp1_k5_100000', 'sp1_k5_5000', 'sp1_k5_10000', 'sp1_k5_150000', 'sp1_k5_200000', 'sp2_k5', 'sp_k5_30000', 'sp_k5_20000', 'sp_k5_50000', 'sp_k5_100000', 'sp_k5_5000', 'sp_k5_10000', 'sp_k5_150000', 'sp_k5_200000'])

In [30]:
CV_METHODS = ['random', 'block', 'sp1']
CV_METHODS = ['block_k5_10000', 'block_k5_100000', 'block_k5_30000', 'block_k5_5000', 'block_k5_50000', 'random', 'sp1_k5_10000', 'sp1_k5_100000', 'sp1_k5_30000', 'sp1_k5_5000', 'sp1_k5_50000']
# TODO: SELECT CROSS VALIDATION TECHNIQUES TO ADOPT 
CV_METHODS = [ 'sp_k5_20000' ]

## parameters
#  [512, 256, 128 , 64] , [1024, 512, 256 , 128]
param_grid_mlp = {
 'nb_neurones_by_layer' : [[64, 32], [ 128, 64, 32] , [ 256, 128 , 64],  [256, 128 , 64, 32]] ,
 'lr' : [0.001, 0.0005],
 'dropout_rate' :  [0, 0.1, 0.2],
 'hidden_activation': ['relu', 'elu', 'tanh'],
 'output_activation' : ['linear'],
 'batch_size' : [16],
 'epochs' : [500]
}

param_grid_mlp ={
     'nb_neurones_by_layer' : [[128, 64, 32]	] ,
 'lr' : [0.0010],
 'dropout_rate' :  [0.1],
 'hidden_activation': ['relu'],
 'output_activation' : ['linear'],
 'batch_size' : [16],
 'epochs' : [100]
}

soil_properties_options = [['pH'], ['CE'], ['CEC'], ['OM'], ['A%'], ['L%'], ['S%']]


## Training MLP -- sweeps every CV strategy generated above (random, block, sp1),
## reusing the exact same fold assignment for every soil-property / depth /
## normalization combination, just like trees_experiments.ipynb.
for soil_properties in soil_properties_options:
    for depth, df in dfs.items():
        for normalization in normalizations:
            for cv_method in CV_METHODS:
                folds = folds_by_method[cv_method]
                print(f"Training model with depth: {depth} | normalization: {normalization} | cv_method: {cv_method}")
                custom_cross_validation_2D_DSM_MLP(
                    region=REGION,
                    dataframe=df,
                    param_grid=param_grid_mlp,
                    normalization=normalization,
                    model='MLP',
                    folds=folds,
                    cv_method=cv_method,
                    depth=depth,
                    soil_properties=soil_properties
                )


Training model with depth: all_depths | normalization: StandardScaler | cv_method: sp_k5_20000
{'nb_neurones_by_layer': [128, 64, 32], 'lr': 0.001, 'dropout_rate': 0.1, 'hidden_activation': 'relu', 'output_activation': 'linear', 'batch_size': 16, 'epochs': 100}
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Training model with depth: all_depths | normalization: StandardScaler | cv_method: sp_k5_20000
{'nb_neurones_by_layer': [128, 64, 32], 'lr': 0.001, 'dropout_rate': 0.1, 'hidden_activation': 'relu', 'output_activation': 'linear', 'batch_size': 16, 'epochs': 100}
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Training model with depth: all_depths | normalization: StandardScaler | cv_method: sp_k5_20000

In [16]:
soil_properties_options

[['pH', 'CE', 'CEC', 'OM', 'A%', 'L%', 'S%']]

In [15]:
# Single fixed configuration for a quick smoke-test (no hyperparameter search)
param_grid_mlp = {
    'nb_neurones_by_layer': [[256, 128, 64, 32]],
    'lr': [0.0010],
    'dropout_rate': [0.1],
    'hidden_activation': ['relu'],
    'output_activation': ['linear'],
    'batch_size': [16],
    'epochs': [300]
}

soil_properties_options = [['pH', 'CE', 'CEC', 'OM', 'A%', 'L%', 'S%']]

## Training MLP -- same fold-based CV sweep as above, just with a single config
for soil_properties in soil_properties_options:
    print(soil_properties)

    for depth, df in dfs.items():
        for normalization in normalizations:
            for cv_method in CV_METHODS:
                folds = folds_by_method[cv_method]
                print(f"Training model with depth: {depth} | normalization: {normalization} | cv_method: {cv_method}")
                custom_cross_validation_2D_DSM_MLP(
                    df, param_grid_mlp, normalization, 'MLP',
                    folds=folds, cv_method=cv_method,
                    depth=depth, soil_properties=soil_properties
                )


['pH', 'CE', 'CEC', 'OM', 'A%', 'L%', 'S%']
Training model with depth: all_depths | normalization: StandardScaler | cv_method: sp_k5_20000


TypeError: custom_cross_validation_2D_DSM_MLP() missing 1 required positional argument: 'model'